[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb) [![View in nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb)

# Proteomics

Run PAC, HPS, OrganAge and PAOPAC on synthetic data. These examples demonstrate the API, not biological performance.

The models use human plasma Olink Explore 3072 NPX, a log2 relative-abundance measure. NPX can be negative. Other assays need separate preparation; pyaging does not harmonize platforms.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import pyaging as pya

clock_names = [
    "pac", "hps",
    "organagechronologicalconventional", "organagemortalityconventional",
    "paopac",
]
feature_tables = {name: pya.utils.get_feature_ranges(name) for name in clock_names}
features = {name: table["feature"].tolist() for name, table in feature_tables.items()}
pd.DataFrame({"input_columns": {name: len(names) for name, names in features.items()}})

,input_columns
pac,129
hps,87
organagechronologicalconventional,2916
organagemortalityconventional,2916
paopac,191


## Build aligned inputs

Use each model's exact feature names. The helper aligns age to 32 sample identifiers. Direct `AnnData` construction preserves NaNs; `df_to_adata` would impute them.

In [2]:
rng = np.random.default_rng(20261002)
samples = pd.Index([f"synthetic_{i:02d}" for i in range(32)], name="sample")
age = pd.Series(np.linspace(40.0, 70.0, len(samples)), index=samples, name="age")

def synthetic_npx(columns):
    return pd.DataFrame(
        rng.normal(0.0, 0.7, size=(len(samples), len(columns))),
        index=samples,
        columns=columns,
    )

def with_age(npx, age_years):
    if not npx.index.is_unique or not age_years.index.is_unique:
        raise ValueError("Sample identifiers must be unique")
    if not npx.columns.is_unique or "age" in npx.columns:
        raise ValueError("Resolve duplicate or reserved input columns")
    aligned = age_years.reindex(npx.index).astype(float)
    if not np.isfinite(aligned).all():
        raise ValueError("Every sample needs a finite chronological age in years")
    return npx.assign(age=aligned)

# The table reports units, not evidence of compatibility with a different assay.
feature_tables["pac"].head()

,feature,unit,low,high
0,age,years,0.0,122.5
1,ada2,NPX,-inf,inf
2,adamts13,NPX,-inf,inf
3,adamts16,NPX,-inf,inf
4,adgrg2,NPX,-inf,inf


## PAC and HPS

[PAC](https://github.com/kuo-lab-uchc/PAC) uses 128 proteins and [HPS](https://github.com/kuo-lab-uchc/HPS) uses 86, both plus `age` in years. Supply normalized NPX without further scaling. Keep lowercase identifiers, including separate `nppb` and `ntprobnp` assays.

All columns are required; NaNs propagate. PAC returns mortality-equivalent years. HPS returns the probability of avoiding major disease or death over ten years.

In [3]:
kuo_proteins = sorted((set(features["pac"]) | set(features["hps"])) - {"age"})
kuo_npx = synthetic_npx(kuo_proteins)
kuo_inputs = with_age(kuo_npx, age)
for name in ["pac", "hps"]:
    missing = set(features[name]) - set(kuo_inputs.columns)
    if missing:
        raise ValueError(f"{name}: missing required inputs {sorted(missing)}")

kuo_adata = ad.AnnData(kuo_inputs)
pya.pred.predict_age(kuo_adata, ["pac", "hps"], verbose=False)
assert np.isfinite(kuo_adata.obs[["pac", "hps"]]).all().all()
assert kuo_adata.obs["hps"].between(0, 1).all()
kuo_adata.obs[["pac", "hps"]].head()

,pac,hps
sample,,
synthetic_00,50.940487,0.906109
synthetic_01,48.538009,0.930831
synthetic_02,31.523923,0.920229
synthetic_03,58.111728,0.824064
synthetic_04,43.577574,0.944730


## OrganAge

These examples use the full Explore 3072 models' first fold. Keep case-sensitive protein symbols and NPX values. Missing columns contribute zero; NaNs propagate.

For organ-specific predictions, choose a clock such as `organagechronologicalbrain`. For other assays, follow the [authors' preparation guidance](https://github.com/ludgergoeminne/organAging#3-other-data-types).

In [4]:
organ_clocks = ["organagechronologicalconventional", "organagemortalityconventional"]
organ_proteins = sorted(set().union(*(features[name] for name in organ_clocks)))
organ_npx = synthetic_npx(organ_proteins)
organ_adata = ad.AnnData(organ_npx)
pya.pred.predict_age(organ_adata, organ_clocks, verbose=False)
assert np.isfinite(organ_adata.obs[organ_clocks]).all().all()
organ_adata.obs[organ_clocks].head()

,organagechronologicalconventional,organagemortalityconventional
sample,,
synthetic_00,61.203524,-0.035365
synthetic_01,52.591469,-0.304399
synthetic_02,61.347902,-0.249981
synthetic_03,51.681665,0.232191
synthetic_04,64.548275,-0.050437


## PAOPAC

Pass the full cohort with `age` in years and NPX values. PAOPAC applies exponentiation, cohort scaling and LOWESS correction internally. Predictions depend on cohort membership, regardless of `batch_size`.

Keep NPX NaNs for the model's missing-value handling. This example includes one NaN and omits optional `TDI`.

In [5]:
paopac_proteins = [name for name in features["paopac"] if name not in {"age", "TDI"}]
paopac_npx = synthetic_npx(paopac_proteins)
paopac_npx.iloc[0, 0] = np.nan
paopac_adata = ad.AnnData(with_age(paopac_npx, age))
pya.pred.predict_age(paopac_adata, "paopac", batch_size=7, verbose=False)
assert np.isfinite(paopac_adata.obs["paopac"]).all()
assert np.isnan(paopac_adata.X[0, 0])  # input NaNs remain unchanged
paopac_adata.obs["paopac"].head()

sample
synthetic_00    36.037396
synthetic_01    41.096683
synthetic_02    44.507700
synthetic_03    47.468443
synthetic_04    45.039386
Name: paopac, dtype: float64

## Inspect coverage and units

Check absent features and supplied NaNs separately. `percent_na` counts only absent features. PAOPAC's missing `TDI` is optional.

In [6]:
outputs = {
    "pac": (kuo_adata, "mortality-equivalent years"),
    "hps": (kuo_adata, "10-year endpoint-free probability"),
    organ_clocks[0]: (organ_adata, "years"),
    organ_clocks[1]: (organ_adata, "relative natural-log mortality hazard"),
    "paopac": (paopac_adata, "cohort-corrected years"),
}
coverage = pd.DataFrame([
    {
        "clock": name,
        "output_unit": unit,
        "absent_features_percent": obj.uns[f"{name}_percent_na"],
        "absent_features": obj.uns[f"{name}_missing_features"],
    }
    for name, (obj, unit) in outputs.items()
]).set_index("clock")
coverage

,output_unit,absent_features_percent,absent_features
clock,,,
pac,mortality-equivalent years,0.00000,[]
hps,10-year endpoint-free probability,0.00000,[]
organagechronologicalconventional,years,0.00000,[]
organagemortalityconventional,relative natural-log mortality hazard,0.00000,[]
paopac,cohort-corrected years,0.52356,[TDI]


In [7]:
pd.Series({
    "PAC/HPS supplied NPX NaNs": int(kuo_npx.isna().sum().sum()),
    "OrganAge supplied NPX NaNs": int(organ_npx.isna().sum().sum()),
    "PAOPAC supplied NPX NaNs": int(paopac_npx.isna().sum().sum()),
})

PAC/HPS supplied NPX NaNs     0
OrganAge supplied NPX NaNs    0
PAOPAC supplied NPX NaNs      1
dtype: int64

## Get citations

Citations and clock-specific notes are in `adata.uns["<clock_name>_metadata"]`. See the [source notes](https://github.com/lucascamillomd/pyaging/blob/main/docs/development/proteomic-source-follow-up.md) for implementation details and model availability.

In [8]:
{
    name: obj.uns[f"{name}_metadata"]["doi"]
    for name, (obj, _) in outputs.items()
}

{'pac': 'https://doi.org/10.1111/acel.14195',
 'hps': 'https://doi.org/10.1073/pnas.2414086122',
 'organagechronologicalconventional': 'https://doi.org/10.1016/j.cmet.2024.10.005',
 'organagemortalityconventional': 'https://doi.org/10.1016/j.cmet.2024.10.005',
 'paopac': 'https://doi.org/10.64898/2026.04.24.720503'}